# autoresearch on a free Colab GPU

<a href="https://colab.research.google.com/github/aroughidea/autoresearch-starter/blob/main/cloud/autoresearch_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
&nbsp;*(made your own copy via "Use this template"? Edit this badge URL — and `COLAB_REPO_URL` below — to point at your username)*

**What this is:** you will train a small GPT language model *from scratch* on a free
Google Colab T4 GPU, then act as the researcher: change one thing in the training
code, retrain (every run gets exactly 5 minutes of training time), and keep or
discard the change based on a single score. This is the same research loop the full
kit runs with an autonomous coding agent — here **you** are the agent, and you need
no hardware and no paid accounts.

**Time to your first trained model: about 30 minutes** (setup + data download + one
training run), all inside this notebook.

### Free-tier reality check (read once, it is short and honest)

- Free Colab assigns an **NVIDIA T4 (16 GB)** — a 2018 datacenter GPU. It works, but
  it is several times slower than the desktop RTX cards this project was tuned on.
  The training budget is fixed *time*, not fixed compute, so your scores will be
  **worse than any numbers you see in the repo's history**. That is expected.
  Compare your runs only against your own T4 baseline.
- Treat free sessions as **1–3 hour sittings**. Colab can disconnect you at any
  time, and a disconnect wipes the VM — files, dataset cache, everything. Section 6
  shows how to push your results out before that happens; do not skip it.
- One experiment takes roughly **8–10 minutes** of wall clock on a T4 (5 min
  training + startup + evaluation), so a sitting fits about **6–20 experiments**.
  That is a genuinely good afternoon of research. It is **not** the place for
  unattended overnight runs.
- Want overnight autonomous runs, faster GPUs, or the full agent experience? The
  **[kit README](https://github.com/aroughidea/autoresearch-starter#readme)** covers
  the local-GPU and rented-pod paths.

### Map of this notebook

1. GPU check → 2. Setup → 3. Data prep → 4. **Manual researcher mode** (the default
path) → 5. Agent mode (optional, advanced) → 6. Save your work.

Run cells top to bottom. Cells marked **EDIT ME** are the ones you change.

## 1 — GPU check

Colab does not give you a GPU unless you ask. Before anything else:
**Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save.**

Then run the next cell. If the assertion fails, do the menu step above and rerun the
cell (a runtime change restarts the session, which is fine — nothing is installed yet).

In [ ]:
# What GPU did Colab give us?
!nvidia-smi

# Friendly check using Colab's preinstalled torch (the project's own torch comes later).
import torch

assert torch.cuda.is_available(), (
    "No GPU visible. Fix: menu bar -> Runtime -> Change runtime type -> "
    "Hardware accelerator: T4 GPU -> Save. Then rerun this cell."
)
print(f"GPU visible to torch: {torch.cuda.get_device_name(0)}")

## 2 — Setup: your repo copy + uv

A two-minute detour that pays off at the end: create **your own copy** of the
starter repo on GitHub — top-right of
[aroughidea/autoresearch-starter](https://github.com/aroughidea/autoresearch-starter):
**Use this template → Create a new repository**. Then point `COLAB_REPO_URL` in the
next cell at your copy.

The notebook also works against the upstream repo unchanged — you just will not be
able to push your results back in Section 6.

In [ ]:
# EDIT ME (recommended): replace with YOUR copy of the starter repo, created on
# GitHub via "Use this template" at https://github.com/aroughidea/autoresearch-starter
# Leaving the default works, but Section 6 cannot push your results to it.
COLAB_REPO_URL = "https://github.com/aroughidea/autoresearch-starter"

REPO_DIR = "/content/autoresearch"

import os
if not os.path.isdir(REPO_DIR):
    !git clone {COLAB_REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
# Install uv (the project's package manager) and put it on PATH for this session.
import os
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ["PATH"]
!uv --version

### Install the project's dependencies

`uv sync` reads the repo's lockfile and `.python-version` (Python 3.14), downloads a
standalone interpreter for this VM, and installs the exact pinned dependency set —
including a torch wheel that is over a gigabyte. Expect **2–5 minutes**; it is quiet
for long stretches while downloading.

In [ ]:
# Creates ./.venv with Python 3.14 + the locked dependencies (2-5 minutes).
!uv sync

### The torch pin, and why we verify it before training

The project pins `torch==2.9.1` built against **CUDA 12.8** (`cu128`) — a pin chosen
for modern desktop RTX cards. Colab's T4 is a 2018 Turing GPU, and Colab's driver
image changes underneath us over time. Rather than assume compatibility, the next
cell **proves it**: it imports the *project's* torch (from `.venv`, not Colab's
preinstalled one), checks that CUDA is visible, and runs a real matmul on the GPU.

- Prints **"Torch verification passed"** → skip the fallback cell and go to Section 3.
- **Fails** (typical symptoms: `no kernel image is available for execution on the
  device`, or a `CUDA error` at startup) → un-comment the fallback cell below it,
  run it, then re-run the verification cell until it passes.

In [ ]:
# GATE: verify the project's pinned torch actually works on this GPU.
import pathlib
import subprocess

check = r'''
import torch
print("torch:", torch.__version__)
assert torch.cuda.is_available(), "project torch cannot see the GPU"
a = torch.randn(1024, 1024, device="cuda", dtype=torch.float16)
checksum = (a @ a).float().abs().sum().item()
torch.cuda.synchronize()
print(f"CUDA matmul OK on {torch.cuda.get_device_name(0)} (checksum {checksum:.3e})")
'''
pathlib.Path("/content/torch_check.py").write_text(check)

r = subprocess.run(["uv", "run", "python", "/content/torch_check.py"],
                   capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
assert r.returncode == 0, (
    "Torch verification FAILED. Un-comment the fallback cell below, run it, "
    "then re-run this cell until it passes."
)
print("Torch verification passed - skip the fallback cell and continue to Section 3.")

In [ ]:
# ------------- FALLBACK: only if the verification cell above FAILED -------------
# Swaps the pinned cu128 torch inside the project venv for a cu126 build, which
# supports older GPU/driver stacks like Colab's T4. Un-comment ALL lines and run.
#
# import os
# os.environ["UV_NO_SYNC"] = "1"   # crucial: otherwise every later `uv run ...`
#                                  # quietly re-installs the pinned cu128 wheel
# !uv pip install torch --index-url https://download.pytorch.org/whl/cu126
#
# Now RE-RUN the verification cell above. If cu126 ALSO fails, the last resort is
# to skip uv entirely and use Colab's preinstalled torch with the system Python:
#     !pip install rustbpe tiktoken pyarrow
# ...then for the rest of the notebook use `!python prepare.py` and
# `!python train.py` wherever a cell says `!uv run prepare.py` / `!uv run train.py`.
# --------------------------------------------------------------------------------

## 3 — Data prep (~650 MB download, one-time per session)

The next cell downloads the TinyStories dataset (a single parquet file, roughly
650 MB) and trains a small BPE tokenizer on it. Results are cached under
`~/.cache/autoresearch`, so re-running it later *in this session* is instant. The
cache dies with the VM, though — every fresh Colab session pays the download again.
Expect a few minutes.

In [ ]:
!uv run prepare.py

## 4 — Manual researcher mode (the default path)

The full kit points an autonomous coding agent at `program.md` and lets it run
experiments unattended. That whole playbook compresses to five lines, and today you
execute them yourself:

1. Pick **one** change to `train.py` (one hyperparameter, one idea).
2. Run `uv run train.py` — it always trains for exactly 5 minutes, then evaluates.
3. Read `val_bpb` — validation **bits per byte**, how well the model compresses
   held-out text. **Lower is better.**
4. Better than your best? **Keep** the change. Worse? **Set it back.**
5. Log every run — keeps, discards, and crashes — in `results.tsv`. Go to 1.

That is the entire research loop. Everything below is tooling for it. No extra
accounts, no agent, no API keys — just you and a GPU.

In [ ]:
# Researcher's toolkit -- run this cell once per session. It defines:
#   set_hparam(name, value)        rewrite ONE top-level constant in train.py
#   show_hparams()                 list tunable constants and current values
#   log_result(status, desc)       append a row to results.tsv + print scoreboard
#   scoreboard()                   print all logged runs, best first
#   reset_results()                start a fresh results.tsv (header only)

import datetime
import pathlib
import re
import subprocess

TRAIN = pathlib.Path("train.py")
RESULTS = pathlib.Path("results.tsv")
TSV_HEADER = "timestamp\tcommit\tval_bpb\tmemory_gb\tstatus\tdescription\n"


def set_hparam(name, value):
    """Rewrite the `NAME = <value>` line in train.py, preserving any trailing comment."""
    src = TRAIN.read_text()
    pattern = re.compile(
        rf"(?m)^{re.escape(name)}(\s*=\s*)([^#\r\n]+?)(\s*)(#[^\r\n]*)?$"
    )
    hits = list(pattern.finditer(src))
    if len(hits) != 1:
        raise ValueError(
            f"Expected exactly one top-level `{name} = ...` line in train.py, "
            f"found {len(hits)}. Run show_hparams() to see valid names."
        )
    m = hits[0]
    old = m.group(2).strip()
    comment = ("  " + m.group(4)) if m.group(4) else ""
    TRAIN.write_text(
        src[: m.start()] + f"{name}{m.group(1)}{value!r}{comment}" + src[m.end():]
    )
    print(f"train.py: {name} = {old}  ->  {value!r}")


def show_hparams():
    lines = TRAIN.read_text().splitlines()
    starts = [i for i, l in enumerate(lines) if l.strip() == "# Hyperparameters"]
    for line in lines[starts[0] if starts else 0:]:
        if line.startswith("def "):
            break
        if re.match(r"^[A-Z][A-Z0-9_]*\s*=", line):
            print(line)


def _git(*args):
    try:
        return subprocess.run(
            ["git", *args], capture_output=True, text=True
        ).stdout.strip()
    except OSError:
        return ""


def scoreboard():
    if not RESULTS.exists():
        print("results.tsv not found -- nothing logged yet.")
        return
    rows = [ln.split("\t") for ln in RESULTS.read_text().splitlines()[1:] if ln.strip()]
    rows = [r for r in rows if len(r) == 6]
    runs = sorted((r for r in rows if r[4] != "crash"), key=lambda r: float(r[2]))
    crashes = [r for r in rows if r[4] == "crash"]
    print(f"{'val_bpb':>10}  {'mem_gb':>6}  {'status':<8}  description")
    print("-" * 76)
    for i, (ts, commit, bpb, mem, status, desc) in enumerate(runs + crashes):
        best = "  <-- best" if runs and i == 0 else ""
        print(f"{bpb:>10}  {mem:>6}  {status:<8}  {desc}{best}")


def log_result(status, description, log_path="run.log"):
    """Append one experiment to results.tsv (tab-separated), then print the scoreboard."""
    assert status in ("keep", "discard", "crash"), \
        "status must be 'keep', 'discard', or 'crash'"
    description = " ".join(str(description).split())  # tabs/newlines would break the TSV
    log = pathlib.Path(log_path)
    text = log.read_text(errors="replace") if log.exists() else ""
    bpb = re.search(r"(?m)^val_bpb:\s+([0-9.]+)", text)
    vram = re.search(r"(?m)^peak_vram_mb:\s+([0-9.]+)", text)
    if bpb is None and status != "crash":
        print("No val_bpb found in run.log -> logging this run as a crash instead.")
        status = "crash"
    val_bpb = float(bpb.group(1)) if bpb else 0.0
    mem_gb = float(vram.group(1)) / 1024 if vram else 0.0
    timestamp = datetime.datetime.now().astimezone().isoformat(timespec="seconds")
    # Use the commit hash only when the working tree is clean; otherwise write 'colab'
    # (set_hparam edits are uncommitted, so HEAD would misrepresent what actually ran).
    commit = _git("rev-parse", "--short=7", "HEAD") if not _git("status", "--porcelain") else "colab"
    if not RESULTS.exists() or not RESULTS.read_text().strip():
        RESULTS.write_text(TSV_HEADER)
    with RESULTS.open("a", newline="") as f:
        f.write(f"{timestamp}\t{commit or 'colab'}\t{val_bpb:.6f}\t{mem_gb:.1f}\t{status}\t{description}\n")
    print(f"logged: {status}  val_bpb={val_bpb:.6f}  mem={mem_gb:.1f}GB  {description}\n")
    scoreboard()


def reset_results():
    """Start a fresh results.tsv. Use once if your copy shipped with desktop-GPU history."""
    RESULTS.write_text(TSV_HEADER)
    print("results.tsv reset to header only.")


print("Toolkit ready. Current tunable constants in train.py:\n")
show_hparams()

### 4a — Establish the baseline

The first run is always the untouched code — that number is your reference. Training
is a fixed 5-minute budget; with startup, batch-size autotuning, and evaluation the
cell takes **~8–10 minutes on a T4**. Output streams live below and is also saved to
`run.log`.

What to expect on a T4 (all normal):

- The script uses its **compatibility runtime profile** (the T4 is not one of the
  desktop RTX cards the kit tiers were tuned for): float16 autocast plus activation
  checkpointing.
- `mfu_percent` prints `n/a` — the script does not know the T4's peak FLOPS.
- If your copy of the repo shipped with rows already in `results.tsv`, those came
  from a much faster desktop GPU. Run `reset_results()` first for a clean
  scoreboard, or just mentally ignore the old rows — only compare T4 against T4.

In [ ]:
# Baseline training run (~8-10 min on a T4). Output also goes to run.log.
!uv run train.py 2>&1 | tee run.log

In [ ]:
# The two numbers that matter, straight from the log:
!grep -E "^val_bpb:|^peak_vram_mb:" run.log || echo "No summary found -> the run crashed. Inspect it with: tail -n 40 run.log"

In [ ]:
# Log the baseline. (If the cell above said the run crashed, change this to
# log_result("crash", "baseline crashed - see run.log") and re-run the training cell.)
log_result("keep", "baseline (Colab T4, untouched train.py)")

### 4b — Your first experiment (EDIT ME)

Change exactly **one** thing, then retrain. The `set_hparam` helper rewrites a single
constant line in `train.py` for you. A menu of starting points, roughly safest-first:

| Constant | Baseline | Try | Why it might matter |
|---|---|---|---|
| `MATRIX_LR` | `0.05` | `0.04`, `0.045`, `0.06` | Muon learning rate for the weight matrices — the classic first knob |
| `WARMDOWN_RATIO` | `0.45` | `0.35`, `0.55` | Fraction of the run spent decaying the learning rate |
| `WEIGHT_DECAY` | `0.1` | `0.05`, `0.2` | Regularization strength |
| `ADAM_BETAS` | `(0.8, 0.95)` | `(0.9, 0.95)` | Momentum for the Adam-family parameters |
| `WINDOW_PATTERN` | `'SSSL'` | `'SSSS'`, `'LLLL'` | Attention window pattern per layer group: `L` = full context, `S` = half |
| `DEPTH` | `6` | `5`, `7` | Deeper = smarter per step but fewer steps in 5 minutes; `7` may be slow or OOM-prone on a T4 |

Strings need quotes — `set_hparam("WINDOW_PATTERN", "SSSS")` — numbers do not.

Beyond single constants: open `train.py` directly (folder icon in the left sidebar →
`autoresearch/train.py`) and edit anything — optimizer logic, architecture, the
training loop. Architecture surgery is legal. This is research.

In [ ]:
# ============================ EDIT ME ============================
# ONE change per experiment. Run this cell, then the training cell
# below it. Baseline values are in the table above.

set_hparam("MATRIX_LR", 0.04)

# Other ideas -- swap in ONE of these instead (and put the previous
# constant back to its best value first):
# set_hparam("WARMDOWN_RATIO", 0.55)
# set_hparam("WINDOW_PATTERN", "SSSS")
# set_hparam("DEPTH", 5)
# =================================================================

In [ ]:
# Train with your change (~8-10 min on a T4), then show the score.
!uv run train.py 2>&1 | tee run.log
!echo "----------------------------------------"; grep -E "^val_bpb:|^peak_vram_mb:" run.log || echo "No summary -> crashed. Inspect with: tail -n 40 run.log"

In [ ]:
# ============================ EDIT ME ============================
# Log it honestly, then the scoreboard prints.
#   status:      "keep" if val_bpb beat your best, else "discard"
#                ("crash" if the run died -- log those too, they're data)
#   description: one line, what you changed (and why, if you had a why)
log_result("keep", "MATRIX_LR 0.05 -> 0.04")
# =================================================================

### 4c — The loop, from here on

1. **EDIT** — change one constant in the EDIT-ME cell (or edit `train.py` directly).
2. **RUN** — re-run the training cell (~8–10 min).
3. **LOG** — `log_result("keep"` or `"discard", "what you tried")`.
4. **REVERT on discard** — put the constant back to your best-known value with
   `set_hparam` (the helper printed the old value when you changed it).
5. Repeat until Colab's clock runs out — then **go to Section 6 and save everything**.

Habits that separate research from knob-twiddling:

- **One variable at a time.** Otherwise you cannot attribute the change in score.
- **Respect noise.** Differences of ±0.002 val_bpb can be run-to-run noise at this
  scale. Re-run before believing a tiny win — confirmation costs 10 minutes, and
  believing a false result costs the rest of your session.
- **Log discards and crashes too.** A map of what does not work is half the value
  of a lab notebook.
- **Watch `peak_vram_mb`** when you grow the model — the T4 has ~15 GB usable.
- **Watch the session clock.** Leave ~10 minutes at the end for Section 6.

## 5 — Agent mode (optional, advanced)

Everything you just did by hand — pick a change, run, compare, log — is exactly what
the full kit automates with a CLI coding agent reading `program.md`. You *can* run
that real agent loop from Colab. Read the caveats first: this is a demo path, not
the recommended one.

**What you need (none of it comes with this notebook):**

- **Your own paid agent account** — an Anthropic account or API key for Claude Code,
  or an OpenAI account or API key for Codex CLI.
- **A terminal.** Colab's built-in **Terminal** button (bottom-left) is a Colab Pro
  feature. On the free tier there is no supported terminal; the community workaround
  is `!pip install colab-xterm`, then `%load_ext colabxterm` and `%xterm` in a code
  cell — it works, but it is unofficial and fragile.

**In the terminal** (Node.js is preinstalled on Colab) — these are the same commands
as the kit README, minus the Windows-specific wrappers:

Claude Code:

```bash
npm install -g @anthropic-ai/claude-code
export ANTHROPIC_API_KEY=...   # or run `claude` once and follow its login flow
cd /content/autoresearch
claude "Read program.md, do setup checks, and start a new experiment loop. Log each result in results.tsv."
```

Codex CLI:

```bash
npm install -g @openai/codex
export OPENAI_API_KEY=...      # or run `codex` once and follow its login flow
cd /content/autoresearch
codex exec -s danger-full-access "Read program.md, do setup checks, and start a new experiment loop. Log each result in results.tsv."
```

**Honest caveats:**

- Colab disconnects without warning. A disconnect kills the agent mid-loop and wipes
  everything not pushed to GitHub. Treat this strictly as a **supervised session** —
  stay at the keyboard, push results often (Section 6).
- Authentication in a headless VM is the usual failure point. API-key environment
  variables are far more reliable than browser login flows here.
- Unattended modes (`claude --dangerously-skip-permissions`, `codex -s
  danger-full-access`) let the agent run arbitrary shell commands unreviewed. On a
  throwaway Colab VM the blast radius is small, but know what you are opting into.
- On a T4 each agent iteration still costs ~8–10 GPU-minutes, so a two-hour
  supervised window buys the agent only ~10 experiments. For real agent sessions,
  use the local-GPU or rented-pod paths in the
  [kit README](https://github.com/aroughidea/autoresearch-starter#readme).

## 6 — Save your work before the VM disappears

Two things are worth keeping: your **results and code** (push them to your GitHub
copy) and your **trained model** (download the checkpoint).

### 6a — Push `results.tsv` and `train.py` to your GitHub copy

This only makes sense if `COLAB_REPO_URL` (Section 2) points at **your** copy of the
repo, not the upstream template.

You need a GitHub **personal access token** with write access to that repo. Best
practice: a *fine-grained* token scoped to just that one repository with the
**Contents: Read and write** permission — GitHub → Settings → Developer settings →
Personal access tokens. The cell below asks for it with a hidden prompt.
**Never paste a token into a code cell** — notebooks save their cell contents and
outputs, and pushed notebooks leak pasted secrets.

In [ ]:
# Commit and push results.tsv + train.py to YOUR GitHub copy.
from getpass import getpass
import subprocess

GITHUB_USERNAME = ""  # <-- EDIT: your GitHub username
GIT_EMAIL = ""        # <-- EDIT: any email for the commit record

assert GITHUB_USERNAME and GIT_EMAIL, "Fill in GITHUB_USERNAME and GIT_EMAIL above first."
token = getpass("GitHub personal access token (input stays hidden): ")

!git config user.name "{GITHUB_USERNAME}"
!git config user.email "{GIT_EMAIL}"
!git add results.tsv train.py
!git commit -m "Colab session: manual researcher mode results" || echo "(nothing new to commit)"

origin = subprocess.run(["git", "remote", "get-url", "origin"],
                        capture_output=True, text=True).stdout.strip()
push_url = origin.replace("https://", f"https://{GITHUB_USERNAME}:{token}@", 1)
r = subprocess.run(["git", "push", push_url, "HEAD"], capture_output=True, text=True)
print((r.stdout + r.stderr).replace(token, "***"))  # never let the token reach cell output
print("Push OK." if r.returncode == 0 else
      "Push FAILED - check the message above. Is COLAB_REPO_URL your own copy, "
      "and does the token have Contents: Read and write on it?")

### 6b — Download your model (~60 MB)

`checkpoint_pre_eval.pt` is the model from the **most recent** training run — every
run overwrites it. If your best result was an earlier run, re-apply the best
settings with `set_hparam` and train once more before downloading.

In [ ]:
import os

ckpt = "checkpoint_pre_eval.pt"
assert os.path.exists(ckpt), "No checkpoint found - complete at least one training run first."
print(f"{ckpt}: {os.path.getsize(ckpt) / 1e6:.0f} MB")

from google.colab import files
files.download(ckpt)  # browser download, ~60 MB

### 6c — Chat with your model on any laptop (no GPU needed)

On your own machine — any laptop, **CPU is fine** for a model this small:

1. Clone your copy of the repo and set it up (kit README: install `uv`, `uv sync`).
2. Run `uv run prepare.py` once — `chat.py` needs the tokenizer, which this rebuilds
   (it re-downloads the ~650 MB dataset).
3. Drop your downloaded `checkpoint_pre_eval.pt` into the repo root.
4. Run `uv run chat.py` — a browser UI opens; type a few words and your model
   continues them, token by token.

Temper expectations: it is a tiny model trained for five minutes on
children's-story data, so you will get charmingly wobbly TinyStories prose. But it
learned everything it knows from scratch, in five minutes, on a free GPU, while you
watched — and you measured every change you made to it. That is the whole game,
small.

**A yardstick, and a way to go further.** The reference deployment at
[autoresearch-demo.fly.dev](https://autoresearch-demo.fly.dev/) runs this same
`chat.py` with two checkpoints from a real 16-experiment session — the baseline and
the best result — so you can see what your own model should look like, and preview
the UI before setting anything up locally. It is free, public, and needs no login; it
sleeps when nobody is visiting, so the first page load takes ~12 seconds to wake it
and is quick after that. Hosting your own works the same way: this same `chat.py` in
a container on a CPU-only machine (a ~19M-parameter model needs no GPU to serve). The
recipe is public in the worked-example repo's
[`deploy/`](https://github.com/aroughidea/autoresearch-win-rtx/tree/master/deploy)
folder.

### Where next

- **[Kit README](https://github.com/aroughidea/autoresearch-starter#readme)** —
  local RTX and rented-pod setups, and running the autonomous agent for real.
- **`program.md`** in the repo — the full playbook the agent follows; also the best
  written description of the research loop you just ran by hand.